# HOTSPOT DETECTION USING DBSCAN

Clusters geographic station data to identify high-pollution hotspots using DBSCAN.

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import pickle

## Load Data

In [2]:
print("Loading station dataset...")
df = pd.read_csv("../data/station_day.csv")
df.columns = df.columns.str.strip().str.lower()

Loading station dataset...


## Clean and Pivot

In [3]:
df = df.dropna(subset=["latitude", "longitude"])
df["pollutant_avg"] = pd.to_numeric(df["pollutant_avg"], errors="coerce")
df = df.dropna(subset=["pollutant_avg"])

pivot_df = df.pivot_table(
    index=["city", "station", "latitude", "longitude"],
    columns="pollutant_id",
    values="pollutant_avg",
    aggfunc="mean"
).reset_index()
pivot_df.columns.name = None

## Create Pollution Score

In [4]:
pollutant_cols = pivot_df.columns.difference(["city", "station", "latitude", "longitude"])
pivot_df["pollution_score"] = pivot_df[pollutant_cols].mean(axis=1)
pivot_df = pivot_df.dropna(subset=["pollution_score"])

## Preprocessing and Clustering

In [5]:
features = pivot_df[["latitude", "longitude", "pollution_score"]]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

print("Training DBSCAN hotspot model...")
dbscan = DBSCAN(eps=0.3, min_samples=5)
clusters = dbscan.fit_predict(X_scaled)
pivot_df["cluster"] = clusters

Training DBSCAN hotspot model...


## Save Model

In [6]:
with open("../models/hotspot_dbscan.pkl", "wb") as f:
    pickle.dump(dbscan, f)
with open("../models/hotspot_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("Hotspot model saved successfully.")

Hotspot model saved successfully.


## Show Results

In [7]:
hotspots = pivot_df[pivot_df["cluster"] != -1]
print("Total Hotspot Stations:", hotspots.shape[0])
print(hotspots[["city", "station", "pollution_score", "cluster"]].head(10))

Total Hotspot Stations: 290
         city                                    station  pollution_score  \
1        Agra                   Manoharpur, Agra - UPPCB        26.142857   
2        Agra                        Rohta, Agra - UPPCB        27.000000   
3        Agra                Sanjay Palace, Agra - UPPCB        37.000000   
4        Agra  Sector-3B Avas Vikas Colony, Agra - UPPCB        32.166667   
5        Agra             Shahjahan Garden, Agra - UPPCB        30.666667   
6        Agra                 Shastripuram, Agra - UPPCB        26.428571   
7   Ahmedabad               Chandkheda, Ahmedabad - IITM        48.857143   
11  Ahmedabad                  Rakhial, Ahmedabad - IITM        51.000000   
12  Ahmedabad           SAC ISRO Bopal, Ahmedabad - IITM        49.428571   
13  Ahmedabad       SAC ISRO Satellite, Ahmedabad - IITM        49.857143   

    cluster  
1         0  
2         0  
3         0  
4         0  
5         0  
6         0  
7         1  
11        1 